# Gen AI Evaluation Service

Agent Platform's Gen AI evaluation service empowers you to assess any generative AI model or application according to your specific requirements.

Instead of relying solely on general leaderboards and reports, you can define your own evaluation criteria and directly compare how different models perform against your unique needs, use case and data.

This service allows you to:

- Gain a deeper understanding of model performance. And, Go beyond general metrics and understand how a model handles your specific data and tasks.
- Make informed decisions throughout the development lifecycle. By using evaluations to guide model selection, refine prompt engineering, and optimize model customization.
- Streamline your evaluation workflow by leveraging Agent Platform's integrated tools to easily launch and reuse evaluations as needed.

Essentially, it puts you in control of evaluating generative AI, ensuring the models you choose are the best fit for your specific applications.

## Use Case: Recipe Assistant

Throughout this notebook we will evaluate a single application: a **Recipe Assistant** built on Gemini. The assistant has two jobs:

1. **Summarize recipes** so that users can quickly understand how a dish is made.
2. **Search for recipes** by calling a `search_recipes` tool (function calling) based on a user's request.

We will use the same set of recipes for every evaluation technique so that you can focus on *how* each technique works and *when* to use it.

## Learning Objectives

In this notebook, you will learn:

- How to use Gen AI Evaluation Service
- Different types of evaluation techniques: computation-based, model-based (pointwise and pairwise) and tool use quality metrics
- How to prepare your dataset and get it ready for evaluation
- Analyze and understand the results from evaluation

### Setup

In [1]:
# General
import json
import random
import string

import pandas as pd
from IPython.display import Markdown, display

# Main
from vertexai.evaluation import (
    EvalTask,
    MetricPromptTemplateExamples,
    PairwiseMetric,
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
)

from google import genai
from google.genai import types

### Helper Functions

`display_eval_report` displays the results from the evaluation SDK in a readable format, and `generate_uuid` creates unique names for our experiment runs.

In [2]:
def display_eval_report(eval_result, metrics=None):
    """Display the evaluation results."""

    title, summary_metrics, report_df = eval_result
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T
    if metrics:
        metrics_df = metrics_df.filter(
            [
                metric
                for metric in metrics_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )
        report_df = report_df.filter(
            [
                metric
                for metric in report_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    # Display the title with Markdown for emphasis
    display(Markdown(f"## {title}"))

    # Display the metrics DataFrame
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    # Display the detailed report DataFrame
    display(Markdown("### Report Metrics"))
    display(report_df)


def generate_uuid(length: int = 8) -> str:
    """Generate a uuid of a specified length (default=8)."""
    return "".join(
        random.choices(string.ascii_lowercase + string.digits, k=length)
    )

## Evaluation Process

Agent Platform's Gen AI evaluation service lets you assess any generative model based on your specific needs and criteria.

These are the four steps you will follow to help you evaluate:

1. Define: Tailor metrics to your business goals and choose your evaluation approach (Computation based vs Model Based).
2. Prepare: Create a dataset that reflects your real-world use case.
3. Run: Easily launch evaluations using templates or existing examples. Define your models and create reusable `EvalTasks` within Agent Platform to use in other evaluations.
4. Analyze: Interpret your results and understand how each model performs against your specific criteria.

Let's take a look at these steps one by one


Let's define a function which takes a model name (e.g. `'gemini-3.5-flash'`) and returns a function that calls the specified model and returns the response. We will use it to power our Recipe Assistant in every section.

In [3]:
# Initialize the standard Google GenAI Client
client = genai.Client(vertexai=True, location="global")


def create_model_fn(model_name: str):
    """
    Generates and returns a model_function tailored to the specified model_name.
    """

    def model_function(prompt: str) -> str:
        # Generate content using your target model
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
        )

        # Return the raw text string response
        return response.text

    return model_function


model_func_3_5_flash = create_model_fn("gemini-3.5-flash")

### Recipe Dataset

This is the shared dataset for the whole notebook. `context` holds three recipe articles that the Recipe Assistant will summarize, and `reference` holds a human-written "ground truth" summary for each one. Different evaluation techniques will use different parts of this dataset:

| Technique | Uses `context` | Uses `reference` |
| --- | --- | --- |
| Computation-based metrics | Yes | Yes (required) |
| Model-based pointwise metrics | Yes | No |
| Model-based pairwise metrics | Yes | No |

In [ ]:
instruction = "Summarize the following recipe"

context = [
    "To make a classic spaghetti carbonara, start by bringing a large pot of salted water to a boil. While the water is heating up, cook pancetta or guanciale in a skillet with olive oil over medium heat until it's crispy and golden brown. Once the pancetta is done, remove it from the skillet and set it aside. In the same skillet, whisk together eggs, grated Parmesan cheese, and black pepper to make the sauce. When the pasta is cooked al dente, drain it and immediately toss it in the skillet with the egg mixture, adding a splash of the pasta cooking water to create a creamy sauce.",
    "Preparing a perfect risotto requires patience and attention to detail. Begin by heating butter in a large, heavy-bottomed pot over medium heat. Add finely chopped onions and minced garlic to the pot, and cook until they're soft and translucent, about 5 minutes. Next, add Arborio rice to the pot and cook, stirring constantly, until the grains are coated with the butter and begin to toast slightly. Pour in a splash of white wine and cook until it's absorbed. From there, gradually add hot chicken or vegetable broth to the rice, stirring frequently, until the risotto is creamy and the rice is tender with a slight bite.",
    "To bake a decadent chocolate cake from scratch, start by preheating your oven to 350°F (175°C) and greasing and flouring two 9-inch round cake pans. In a large mixing bowl, cream together softened butter and granulated sugar until light and fluffy. Beat in eggs one at a time, making sure each egg is fully incorporated before adding the next. In a separate bowl, sift together all-purpose flour, cocoa powder, baking powder, baking soda, and salt. Divide the batter evenly between the prepared cake pans and bake for 25-30 minutes, or until a toothpick inserted into the center comes out clean.",
]

reference = [
    "The process of making spaghetti carbonara involves boiling pasta, crisping pancetta or guanciale, whisking together eggs and Parmesan cheese, and tossing everything together to create a creamy sauce.",
    "Preparing risotto entails sautéing onions and garlic, toasting Arborio rice, adding wine and broth gradually, and stirring until creamy and tender.",
    "Baking a decadent chocolate cake requires creaming butter and sugar, beating in eggs, combining the dry ingredients and baking the batter until a toothpick comes out clean.",
]

eval_dataset = pd.DataFrame(
    {
        "context": context,
        "reference": reference,
        "instruction": [instruction] * len(context),
    }
)

eval_dataset

We will use [EvalTask](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) to kick off an Agent Platform Experiment for each evaluation. All runs in this notebook are logged to the same experiment so that you can compare them in one place.

#### Understand the EvalTask class

The EvalTask class is a core component of the Gen AI Evaluation Service SDK framework. It allows you to define and run evaluation jobs against your Gen AI models/applications, providing a structured way to measure their performance on specific tasks. Think of an EvalTask as a blueprint for your evaluation process. Evaluation tasks must contain an evaluation dataset, and a list of metrics to evaluate. Supported metrics are documented on the Generative AI [Define your evaluation metrics page](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval). The dataset can be an `pandas.DataFrame`, Python dictionary or a file path URI and can contain default column names such as `prompt`, `reference`, `response`, and `baseline_model_response`.

- Bring-your-own-response (BYOR): You already have the data that you want to evaluate stored in the dataset. You can customize the response column names for both your model and the baseline model using parameters like response_column_name and baseline_model_response_column_name or through the metric_column_mapping.

- Perform model inference without a prompt template: You have a dataset containing the input prompts to the model and want to perform model inference before evaluation. A column named prompt is required in the evaluation dataset and is used directly as input to the model.

- Perform model inference with a prompt template: You have a dataset containing the input variables to the prompt template and want to assemble the prompts for model inference. Evaluation dataset must contain column names corresponding to the variable names in the prompt template. For example, if prompt template is "Instruction: {instruction}, context: {context}", the dataset must contain instruction and context columns.

EvalTask supports extensive evaluation scenarios including BYOR, model inference with Gemini models, 3P models endpoints/SDK clients, or custom model generation functions, using computation-based metrics, model-based pointwise and pairwise metrics. The evaluate() method triggers the evaluation process, optionally taking a model, prompt template, experiment logging configuartions, and other evaluation run configurations. You can view the SDK reference documentation for [Gen AI Evaluation](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation) package for more details.

In this notebook you will see all three modes in action.

In [ ]:
EXPERIMENT_NAME = "recipe-assistant-eval"

## 1. Computation Based Metrics

[Computation based metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) are computed using mathematical formulas to compare the model's output against a **ground truth or reference**. Commonly used computation-based metrics include ROUGE and BLEU. The commonly used metrics can be categorized into the following groups:

- Lexicon-based metrics: Use math to calculate the string similarities between LLM-generated results and ground truth, such as `Exact Match` and `ROUGE`.
- Count-based metrics: Aggregate the number of rows that hit or miss certain ground-truth labels, such as `F1-score`, `Accuracy`, and `Tool Name Match`.
- Embedding-based metrics: Calculate the distance between the LLM-generated results and ground truth in the embedding space, reflecting their level of similarity. These include `cosine similarity` and `euclidean distance`

For our Recipe Assistant, let's use computation based metrics to decide **which prompt template produces the best recipe summaries**. Because these metrics need a ground truth, we will use both the `context` and `reference` columns of our recipe dataset.

Now, we'll define different prompt templates to evaluate. This list can include all the different prompt templates you want to experiment with. For the purpose of demostration, we'll use two different prompt templates below.

In [ ]:
prompt_templates = [
    "Instruction: {instruction}. Article: {context}. Summary:",
    "Article: {context}. Complete this task: {instruction}, in one sentence. Summary:",
]

Next, we will define the different metrics we want to measure for this task. There are different metrics that are defined by Agent Platform based on the task. You can take a look at the [documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) to see the different metrics. And, take a look at the [API documentation](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) to refer to the right string values for each.

In [ ]:
metrics = [
    "rouge_1",
    "rouge_l_sum",
    "bleu",
    "safety",
]

Now, that we have configured all the parameters, we are ready to start evaluating. Here we use **model inference with a prompt template**: the EvalTask fills each prompt template with the `instruction` and `context` columns, calls our model and scores the response against `reference`.

In [ ]:
summarization_eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=metrics,
    experiment=EXPERIMENT_NAME,
)

Please note: The cell below takes 10-15 mins to finish executing. 

In [ ]:
run_id = generate_uuid()
eval_results = []


for i, prompt_template in enumerate(prompt_templates):
    experiment_run_name = f"eval-prompt-engineering-{run_id}-prompt-{i}"

    eval_result = summarization_eval_task.evaluate(
        prompt_template=prompt_template,
        experiment_run_name=experiment_run_name,
        model=model_func_3_5_flash,
    )

    eval_results.append(
        (f"Prompt #{i}", eval_result.summary_metrics, eval_result.metrics_table)
    )

In [ ]:
for eval_result in eval_results:
    display_eval_report(eval_result)

## 2. Model Based Metrics

Computation based metrics require a reference, and they only measure word overlap, not whether a summary is actually *good*. When you do not have ground truths, or want to measure qualitative properties, you can evaluate your model using another LLM. This technique is called **Model based evaluation**. Model-based metrics assesses your candidate model against a judge model. The judge model for most use cases is Gemini, but you can also use models such as [MetricX](https://github.com/google-research/metricx) or [COMET](https://huggingface.co/Unbabel/wmt22-comet-da) for translation use cases.

You can measure model-based metrics pairwise or pointwise:

- **Pointwise metrics**: Let the judge model assess the candidate model's output based on the evaluation criteria. For example, the score could be 0 to 5, where 0 means the response does not fit the criteria, while 5 means the response fits the criteria well.

- **Pairwise metrics**: Let the judge model compare the responses of two models and pick the better one. This is often used when comparing a candidate model with the baseline model. Pairwise metrics are only supported with Gemini as a judge model.

### 2a. Pointwise Metrics

Now that we have picked a prompt template, we want to rate the recipe summaries on two qualitative criteria that matter for a Recipe Assistant: **fluency** and **actionability** (can a home cook follow it?). We will define a custom metric called `text_quality` using those two criteria.

In [ ]:
# Your own definition of text_quality.
metric_prompt_template = PointwiseMetricPromptTemplate(
    criteria={
        "fluency": "Sentences flow smoothly and are easy to read, avoiding awkward phrasing or run-on sentences. Ideas and sentences connect logically, using transitions effectively where needed.",
        "actionability": "The summary captures the key ingredients and cooking steps in a logical order so that a home cook could understand how to prepare the dish.",
    },
    rating_rubric={
        "2": "The response performs well on both criteria.",
        "1": "The response is somewhat aligned with both criteria",
        "0": "The response falls short on both criteria",
    },
)

text_quality = PointwiseMetric(
    metric="text_quality",
    metric_prompt_template=metric_prompt_template,
)

Let's take a look at what this `text_quality` metric looks like.

In [ ]:
print(text_quality.metric_prompt_template)

This time we will use the **bring-your-own-response (BYOR)** mode. We first generate summaries of the same recipes with our model of interest (`gemini-3.5-flash`) and store them in a `response` column. Notice that no `reference` is needed.

In [ ]:
prompt_template = "Instruction: {instruction}. Article: {context}. Summary:"

responses = [
    model_func_3_5_flash(
        prompt_template.format(instruction=instruction, context=article)
    )
    for article in context
]
responses

In [ ]:
pointwise_eval_dataset = pd.DataFrame(
    {
        "response": responses,
    }
)

In [ ]:
pointwise_eval_task = EvalTask(
    dataset=pointwise_eval_dataset,
    metrics=[text_quality],
    experiment=EXPERIMENT_NAME,
)

pointwise_eval_results = pointwise_eval_task.evaluate(
    experiment_run_name=f"eval-pointwise-{generate_uuid()}"
)

You can view the `summary_metrics` aggregated over all 3 recipes here.

In [ ]:
pointwise_eval_results.summary_metrics

To get the score for each recipe based on the criteria you defined in `text_quality` you can access the `metrics_table`.

In [ ]:
pointwise_eval_results.metrics_table

You can see the rationale behind the judge model's score for each summary in the `text_quality/explanation` column.

### 2b. Pairwise Metrics - Compare Models Side-by-Side (SxS)

Next, we want to decide **which model should power our Recipe Assistant**: `gemini-3.1-flash-lite` or `gemini-3.5-flash`. You can evaluate the recipe summaries from two different models using pairwise model evaluation and side-by-side comparison.

To directly compare two models, you can define a `PairwiseMetric` within an `EvalTask` run. This approach allows for a head-to-head assessment of the models' performance. We will reuse the `context` and `instruction` columns of our recipe dataset and let the EvalTask run inference for both models.

In [ ]:
pairwise_eval_dataset = eval_dataset[["context", "instruction"]]

Once we have prepared the evaluation dataset we will define the two models that we want to compare. `model_a` will be `gemini-3.1-flash-lite` and `model_b` will be `gemini-3.5-flash`.

In [ ]:
model_a = create_model_fn("gemini-3.1-flash-lite")
model_b = create_model_fn("gemini-3.5-flash")

Similar to how we defined the evaluation criteria for pointwise evaluation above, you could customize your evaluation prompt. However, to save time, Agent Platform provides [predefined evaluation prompts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/metrics-templates) in `MetricPromptTemplateExamples` you could use. In this use case, we are going to be evaluating the text quality between the two models. `pairwise_text_quality` will use the following criteria to evaluate the model responses.

```
STEP 1: Analyze Response A based on all the Criteria provided, including Coherence, Fluency, Instruction following, Groundedness, and Verbosity. Provide assessment according to each criterion.
STEP 2: Analyze Response B based on all the Criteria provided, including Coherence, Fluency, Instruction following, Groundedness, and Verbosity. Provide assessment according to each criterion 
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment of each criterion 
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field, justifying your choice by highlighting the specific strengths and weaknesses of each response in terms of Text Quality
```

In [ ]:
# Create a "Pairwise Text Quality" metric
text_quality_prompt_template = MetricPromptTemplateExamples.get_prompt_template(
    "pairwise_text_quality"
)

pairwise_text_quality_metric = PairwiseMetric(
    metric="pairwise_text_quality",
    metric_prompt_template=text_quality_prompt_template,
    baseline_model=model_a,
)

Once we have defined the dataset and the evaluation criteria, we are ready to kick off the evaluation job.

Please note: the cell below is going to take 10-15 mins to finish execution.

In [ ]:
pairwise_text_quality_eval_task = EvalTask(
    dataset=pairwise_eval_dataset,
    metrics=[pairwise_text_quality_metric],
    experiment=EXPERIMENT_NAME,
)

# Specify candidate model for pairwise comparison
pairwise_text_quality_result = pairwise_text_quality_eval_task.evaluate(
    model=model_b,
    prompt_template=prompt_template,
    experiment_run_name=f"eval-pairwise-{generate_uuid()}",
)

Once the above experiment runs, you can view the results using the `display_eval_report` helper function.

In [ ]:
display_eval_report(
    (
        "Side-by-side EvalTask",
        pairwise_text_quality_result.summary_metrics,
        pairwise_text_quality_result.metrics_table,
    )
)

## 3. Tool Use Quality Metrics

Besides summarizing recipes, our Recipe Assistant also needs to find recipes for users. It does this by interacting with an external recipe database through **Function Calling**.

The Gen AI Evaluation SDK provides specialized computation-based metrics to assess tool use accuracy:
- `tool_call_valid`: Verifies that the model produced a syntactically valid function call.
- `tool_name_match`: Checks whether the model selected the correct tool/function name.
- `tool_parameter_key_match`: Evaluates whether all expected parameter keys were included.
- `tool_parameter_kv_match`: Evaluates whether both parameter keys and their corresponding values match the ground-truth reference.

Suppose our assistant has access to a tool named `search_recipes` that searches a recipe repository using parameters:
- `recipe_name` (string): Title of the dish (e.g., "Spaghetti Carbonara", "Wild Mushroom Risotto").
- `cuisine` (string): Cuisine style or tradition (e.g., "Italian").
- `servings` (string): Desired number of servings.
- `dietary_restriction` (string): Dietary preferences (e.g., "gluten-free", "vegetarian").
- `max_prep_time` (string): Preparation time limit (e.g., "30 minutes").
- `difficulty` (string): Cooking difficulty (e.g., "easy", "medium", "hard").

In [ ]:
tool_metrics = [
    "tool_call_valid",
    "tool_name_match",
    "tool_parameter_key_match",
    "tool_parameter_kv_match",
]

### 3a. Evaluate a Bring-Your-Own-Response (BYOR) Dataset

You can evaluate tool use quality on pre-recorded model responses and expected ground-truth references.

Each tool call is formatted as a JSON string:
`{"content": "", "tool_calls": [{"name": "<function_name>", "arguments": {<key>: <value>}}]}`

Let's test four representative scenarios in our recipe assistant:
1. **Exact match**: Model correctly calls `search_recipes` for Spaghetti Carbonara with all parameters matching.
2. **Value mismatch**: Model calls `search_recipes` for Carbonara when the user requested Mushroom Risotto.
3. **Missing parameters (key mismatch)**: Model omits dietary restrictions and prep time.
4. **Value variations**: Model extracts different serving sizes and time constraints.

In [ ]:
tool_responses = [
    # 1. Exact match (Spaghetti Carbonara)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 2. Value mismatch (Carbonara instead of Risotto)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 3. Missing parameters (only recipe_name and cuisine)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                    },
                }
            ],
        }
    ),
    # 4. Multiple value differences (different servings, dietary restriction, prep time, difficulty)
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "6",
                        "dietary_restriction": "none",
                        "max_prep_time": "20 minutes",
                        "difficulty": "easy",
                    },
                }
            ],
        }
    ),
]

tool_references = [
    # 1. Expected: Exact match for Carbonara
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 2. Expected: Wild Mushroom Risotto
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Wild Mushroom Risotto",
                        "cuisine": "Italian",
                        "servings": "2",
                        "dietary_restriction": "vegetarian",
                        "max_prep_time": "45 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 3. Expected: Complete parameter set for Carbonara
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
    # 4. Expected: Specific dietary and time constraints
    json.dumps(
        {
            "content": "",
            "tool_calls": [
                {
                    "name": "search_recipes",
                    "arguments": {
                        "recipe_name": "Spaghetti Carbonara",
                        "cuisine": "Italian",
                        "servings": "4",
                        "dietary_restriction": "gluten-free",
                        "max_prep_time": "30 minutes",
                        "difficulty": "medium",
                    },
                }
            ],
        }
    ),
]

tool_eval_dataset = pd.DataFrame(
    {
        "response": tool_responses,
        "reference": tool_references,
    }
)

tool_eval_dataset

Define the `EvalTask` for evaluating tool usage quality across our test cases:

In [ ]:
tool_use_eval_task = EvalTask(
    dataset=tool_eval_dataset,
    metrics=tool_metrics,
    experiment=EXPERIMENT_NAME,
)

In [ ]:
eval_result = tool_use_eval_task.evaluate(
    experiment_run_name=f"eval-tool-use-byor-{generate_uuid()}"
)

display_eval_report(
    (
        "Tool Use Quality Evaluation Metrics (BYOR)",
        eval_result.summary_metrics,
        eval_result.metrics_table,
    )
)

### 3b. Live Model Function Calling and Evaluation with Google GenAI SDK

Now let's test a live Gemini model (`gemini-3.5-flash`) performing function calling using the Google GenAI SDK (`google.genai`), and evaluate its generated tool call against an expected reference.

We will:
1. Define the `search_recipes` tool declaration using `types.FunctionDeclaration` and `types.Tool`.
2. Send a natural language recipe search request to Gemini.
3. Extract and unpack the function call response into the format expected by the evaluation service.
4. Run `EvalTask` to evaluate the model's function calling precision.

In [ ]:
search_recipes_func = types.FunctionDeclaration(
    name="search_recipes",
    description="Search for recipes based on dish name, cuisine, servings, and dietary preferences.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "recipe_name": types.Schema(
                type=types.Type.STRING,
                description="The name of the recipe or dish to search for.",
            ),
            "cuisine": types.Schema(
                type=types.Type.STRING,
                description="The cuisine style or cuisine (e.g., Italian, French, American).",
            ),
            "servings": types.Schema(
                type=types.Type.STRING,
                description="The number of servings to prepare.",
            ),
            "dietary_restriction": types.Schema(
                type=types.Type.STRING,
                description="Dietary restrictions (e.g., gluten-free, vegetarian, dairy-free, none).",
            ),
            "max_prep_time": types.Schema(
                type=types.Type.STRING,
                description="Maximum preparation or cooking time (e.g., 30 minutes).",
            ),
            "difficulty": types.Schema(
                type=types.Type.STRING,
                description="Difficulty level (e.g., easy, medium, hard).",
            ),
        },
        required=[
            "recipe_name",
            "cuisine",
            "servings",
            "dietary_restriction",
            "max_prep_time",
            "difficulty",
        ],
    ),
)

search_recipes_tool = types.Tool(
    function_declarations=[search_recipes_func],
)

In [ ]:
prompt = """I'd like to find a recipe for classic Spaghetti Carbonara for 4 servings that is gluten-free, 
with a max prep time of 30 minutes and medium difficulty in the Italian cuisine tradition.
"""

gemini_response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(tools=[search_recipes_tool]),
)

# Inspect the function calls returned by Gemini
gemini_response.function_calls

In [ ]:
def unpack_response(response):
    """Extracts function calls from a GenerateContentResponse into the eval JSON format."""
    if response.function_calls:
        fc = response.function_calls[0]
        output = {"content": "", "tool_calls": [{"name": fc.name, "arguments": dict(fc.args)}]}
    else:
        fc = response.candidates[0].content.parts[0].function_call
        output = {"content": "", "tool_calls": [{"name": fc.name, "arguments": dict(fc.args)}]}
    return json.dumps(output)


response = unpack_response(gemini_response)
response

In [ ]:
reference_str = json.dumps(
    {
        "content": "",
        "tool_calls": [
            {
                "name": "search_recipes",
                "arguments": {
                    "recipe_name": "Spaghetti Carbonara",
                    "cuisine": "Italian",
                    "servings": "4",
                    "dietary_restriction": "gluten-free",
                    "max_prep_time": "30 minutes",
                    "difficulty": "medium",
                },
            }
        ],
    }
)

live_eval_dataset = pd.DataFrame({"response": [response], "reference": [reference_str]})

In [ ]:
# Expected Tool Call Reference
json.loads(live_eval_dataset.reference[0])

In [ ]:
# Actual Gemini Tool Call Response
json.loads(live_eval_dataset.response[0])

In [ ]:
live_eval_result = EvalTask(
    dataset=live_eval_dataset,
    metrics=tool_metrics,
    experiment=EXPERIMENT_NAME,
).evaluate(experiment_run_name=f"eval-tool-use-live-{generate_uuid()}")

display_eval_report(
    (
        "Gemini Live Tool Use Quality Evaluation Metrics",
        live_eval_result.summary_metrics,
        live_eval_result.metrics_table,
    )
)

All of the evaluation runs in this notebook are logged to the `recipe-assistant-eval` experiment. You can compare them using `display_runs()` or from the [Experiments Tab](https://console.cloud.google.com/agent-platform/experiments/experiments) in the Agent Platform UI.

In [ ]:
summarization_eval_task.display_runs()

## Summary and Key Takeaways

In this notebook, we used the **Gen AI Evaluation Service** to evaluate a single **Recipe Assistant** application with every key evaluation technique:

| Technique | Question answered | Metrics | EvalTask mode |
| --- | --- | --- | --- |
| Computation-based | Which prompt template writes the best recipe summaries? | `rouge_1`, `rouge_l_sum`, `bleu`, `safety` | Inference with prompt template |
| Model-based pointwise | How fluent and actionable are the summaries? | Custom `text_quality` (LLM-as-a-judge) | BYOR |
| Model-based pairwise | Should we use `gemini-3.1-flash-lite` or `gemini-3.5-flash`? | `pairwise_text_quality` | Inference for baseline and candidate models |
| Tool use quality | Does the assistant call `search_recipes` correctly? | `tool_call_valid`, `tool_name_match`, `tool_parameter_key_match`, `tool_parameter_kv_match` | BYOR and live Gemini function calling |

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.